# Taller CUDA - Google Colab

<p align="center">
  <img src="https://raw.githubusercontent.com/Albonire/taller-cuda-unipamplona/main/assets/svg/banner_cuda.svg" width="900">
</p>

Este notebook compila y ejecuta los archivos CUDA del repositorio.

> Recomendado: `Runtime > Change runtime type > GPU`

In [ ]:
!nvidia-smi
!nvcc --version

## Clonar repositorio

Si ya subiste el repo a GitHub, esta celda lo clona dentro de Colab.

Si todavía no lo subiste, puedes cargar la carpeta manualmente desde el panel de archivos de Colab, pero lo recomendado es usar GitHub.

In [ ]:
import os
import pathlib

GITHUB_USER = "Albonire"
REPO_NAME = "taller-cuda-unipamplona"

base = pathlib.Path("/content")
repo_path = base / REPO_NAME

if not repo_path.exists():
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {repo_path}

os.chdir(repo_path)
print("Directorio actual:", os.getcwd())

!find . -maxdepth 2 -type f | sort

In [ ]:
# update repository inside the colab session and show verification
!cd /content/taller-cuda-unipamplona && git pull origin main || echo 'git pull failed: check path or credentials'
echo '\nlast commit (if pull succeeded):'
!cd /content/taller-cuda-unipamplona && git log -1 --pretty=format:'%h %ad %an %s' --date=short
echo '\nfirst lines of scripts/run_all.sh:'
!sed -n '1,80p' /content/taller-cuda-unipamplona/scripts/run_all.sh

## Diagramas del taller

In [ ]:
from IPython.display import SVG, display

display(SVG(filename="assets/svg/sum_three_arrays.svg"))
display(SVG(filename="assets/svg/cuda_memory_flow.svg"))
display(SVG(filename="assets/svg/grid_blocks_threads.svg"))
display(SVG(filename="assets/svg/reduction_tree.svg"))

## Detectar arquitectura CUDA

En Colab normalmente se recibe una NVIDIA T4, que usa `sm_75`.
Esta celda detecta automáticamente la arquitectura.

In [ ]:
import subprocess

def detect_arch():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            check=False
        ).stdout.strip()

        if out:
            cap = out.splitlines()[0].replace(".", "").strip()
            return f"sm_{cap}"
    except Exception:
        pass

    return "sm_75"

arch = detect_arch()
print("Arquitectura detectada:", arch)

## Ejercicio 1: versión CPU y versión CUDA

In [ ]:
!mkdir -p bin

!gcc -O2 src/sum_array.c -o bin/sum_array_cpu
!./bin/sum_array_cpu

!nvcc -O2 -arch={arch} src/sum_array.cu -o bin/sum_array_cuda
!./bin/sum_array_cuda

## Ejecutar todo el taller

In [ ]:
!bash scripts/run_all.sh

## Compilar un ejercicio específico

Ejemplo con suma de vectores:

In [ ]:
FILE = "src/ej04_vector_add.cu"
OUT = "bin/ej04_vector_add_cuda"

!nvcc -O2 -arch={arch} {FILE} -o {OUT}
!./{OUT}